In [ ]:
import numpy as np
import json
import os
from PIL import Image

from orion.utils.misc_utils import *
from orion.utils.o3d_utils import *
import trimesh
import open3d as o3d
from PIL import Image

import os
from pathlib import Path
import cv2
from tqdm import tqdm

import matplotlib.pyplot as plt
import math


In [ ]:
frame = 14
depth_estimator = "metric3d"
original_mesh_loc = "/home/arisrei/research/RPL/ORION/meshes/iphone_long_horizon_boat_18_demo_VLM_rmbg/instant-mesh-large/meshes/iphone_long_horizon_boat_18_demo_VLM_rmbg.obj"
annotation_dir = "/home/arisrei/research/RPL/ORION/annotations/human_demo/iphone_long_horizon_boat_18_demo_VLM"
tgt_obj = 2

original_mesh_parent_dir = Path(original_mesh_loc).parent.absolute()
original_mesh = trimesh.load(original_mesh_loc)
depth_dir = os.path.join(annotation_dir, "FoundationPose", "depth") #/home/arisrei/research/RPL/ORION/annotations/human_demo/mug_no_handle/FoundationPose/depth"

config = json.load(open(os.path.join(annotation_dir, "config.json")))
if depth_estimator != "gt":
    depth = np.load(os.path.join(depth_dir, "all_pred_depth.npy"))[frame]
else:
    with h5py.File(config["original_file"], "r") as demo_file:
        depth = demo_file["data/human_demo/obs/agentview_depth/"][()][frame]

rgb = np.array(Image.open(os.path.join(annotation_dir, "FoundationPose", "rgb", f"rgb_{frame:04d}.png")))
masks = np.load(os.path.join(annotation_dir, "masks.npz"))['arr_0']

with h5py.File(config["original_file"], 'r') as f:
    data_config = json.loads(f['data'].attrs["data_config"])

#intrinsics = np.array([[570.309448, 0.000000, 238.970123],[0.000000, 570.309448, 321.383911],[0.000000, 0.000000, 1.000000]])
intrinsics = np.zeros((3,3))
intrinsics[0,0] = data_config["intrinsics"]['camera_rs_0']['fx']
intrinsics[1,1] = data_config["intrinsics"]['camera_rs_0']['fy']
intrinsics[0,2] = data_config["intrinsics"]['camera_rs_0']['cx']
intrinsics[1,2] = data_config["intrinsics"]['camera_rs_0']['cy']
intrinsics[2,2] = 1

print(intrinsics)

In [ ]:
plane_estimation_kwargs={
    "ransac_n": 3,
    "num_iterations": 1000,
    "distance_threshold": 0.01
}

#scale if needed
depth = depth.astype(np.float32)
depth /= 1000

#get the rotation matrix
pcd_wrapper = O3DPointCloud()
pcd_wrapper.create_from_rgbd(rgb, depth, intrinsics, depth_scale=1, depth_trunc=5)
pcd_wrapper.max_points = 20000
plane_estimation_result = pcd_wrapper.plane_estimation(**plane_estimation_kwargs)
pcd_wrapper.preprocess()

T_xy_plane_align = estimate_rotation(plane_estimation_result["plane_model"], z_up=False)
pcd_wrapper.transform(T_xy_plane_align)
fig = visualize_o3d_point_cloud(pcd_wrapper, offline=True)
display(fig)

# # Extract plane coefficients
# a, b, c, d = plane_estimation_result["plane_model"]

# # Define a grid of points
# x = np.linspace(-.5, .5, 2)
# y = np.linspace(-.5, .5, 2)
# x, y = np.meshgrid(x, y)

# # Compute z-values using the plane equation ax + by + cz + d = 0
# z = (-d - a * x - b * y) / c

# # Add the plane to the figure
# fig.add_trace(go.Surface(x=x, y=y, z=z, colorscale='Viridis', opacity=0.5))

# # Show the figure
# display(fig)



In [ ]:
masks[frame][masks[frame] != tgt_obj] = 0

mask_orig = np.copy(masks[frame])
erode_iter = 4
masks[frame] = cv2.erode(masks[frame].astype(np.uint8), np.ones((3,3), np.uint8), iterations=erode_iter)
depth[masks[frame] != tgt_obj] = 11

#display comparison of original mask and eroded
fig, ax = plt.subplots(1,2)
ax[0].imshow(mask_orig)
ax[1].imshow(masks[frame])
plt.show()



In [ ]:
keypoints = []
for y in range(masks[frame].shape[0]):
    for x in range(masks[frame].shape[1]):
        if masks[frame][y,x] == tgt_obj:
            keypoints.append([y,x])

In [ ]:
#verify that the mask matches the rgb

cpy = np.copy(rgb)
cpy[masks[frame] != tgt_obj] = 0
display(Image.fromarray(cpy))

In [ ]:
# #backproject depths to 3d
pcd_wrapper.create_from_keypoints(keypoints, rgb, depth, intrinsics, depth_scale=1, depth_trunc=5)
#pcd_wrapper.create_from_rgbd(rgb, depth, intrinsics, depth_scale=1, depth_trunc=5)
pcd_wrapper.preprocess()

pcd_wrapper.transform(T_xy_plane_align)
fig = visualize_o3d_point_cloud(pcd_wrapper, offline=True)
display(fig)

pcd_wrapper.remove_outliers(display_removed=True)


In [ ]:
pcd_wrapper.save(os.path.join(depth_dir, "obj_pcd.ply"))
print('saved to', os.path.join(depth_dir, "obj_pcd.ply"))

#open as trimesh
mesh = trimesh.load(os.path.join(depth_dir, "obj_pcd.ply"))

#move mesh to center
# mesh.vertices -= mesh.centroid

# mesh_scene = mesh.scene()
# mesh_scene.add_geometry(trimesh.creation.axis(origin_size=0.01))
# mesh_scene.show()

In [ ]:
to_origin_colmap, extents_colmap = trimesh.bounds.oriented_bounds(mesh, ordered=True)
align_mesh_to_canonical_axes(mesh)
print("Size of mesh along canonical axes:",  mesh.extents)


#mesh.apply_transform(to_origin_colmap)
mesh.export(os.path.join(original_mesh_parent_dir, "obj_pcd.ply"))
print(f"saved to {os.path.join(original_mesh_parent_dir, 'obj_pcd.ply')}")

mesh_scene = mesh.scene()
mesh_scene.add_geometry(trimesh.creation.axis(origin_size=0.01))
display(mesh_scene.show())

In [ ]:
to_origin_original, extents_original = trimesh.bounds.oriented_bounds(original_mesh, ordered=True)
align_mesh_to_canonical_axes(original_mesh)
print("Size of mesh along canonical axes:",  original_mesh.extents)
original_mesh_scene = original_mesh.scene()
original_mesh_scene.add_geometry(trimesh.creation.axis(origin_size=0.25))
original_mesh_scene.show()
#original_mesh.apply_transform(to_origin_original)
#original_mesh.apply_transform(np.linalg.inv(to_origin_original))



In [ ]:
#rotate original mesh to align with the point cloud if needed
print(original_mesh.bounds[1] - original_mesh.bounds[0])
print(mesh.bounds[1] - mesh.bounds[0])
center = original_mesh.centroid

original_mesh_copy = original_mesh.copy()

rot_matrix = trimesh.transformations.rotation_matrix(-math.pi/2, [1, 0, 0], center)
original_mesh_copy.apply_transform(rot_matrix)

# rot_matrix = trimesh.transformations.rotation_matrix(-math.pi/2, [0, 1, 0], center)
# original_mesh_copy.apply_transform(rot_matrix)

# rot_matrix = trimesh.transformations.rotation_matrix(math.pi/2, [0, 0, 1], center)
# original_mesh_copy.apply_transform(rot_matrix)


print(original_mesh_copy.bounds[1] - original_mesh_copy.bounds[0])
print(mesh.bounds[1] - mesh.bounds[0])
original_mesh_copy_scene = original_mesh_copy.scene()
original_mesh_copy_scene.add_geometry(trimesh.creation.axis(origin_size=.25))
original_mesh_copy_scene.show()
#original_mesh.scene().show()

In [ ]:
original_mesh = original_mesh_copy

In [ ]:
#rescale original mesh to have same dimensions as the point cloud
# Get the axis-aligned bounding box (AABB) for mesh A
aabb_pcd = mesh.bounds
length_A_x = aabb_pcd[1][0] - aabb_pcd[0][0]
length_A_y = aabb_pcd[1][1] - aabb_pcd[0][1]
length_A_z = aabb_pcd[1][2] - aabb_pcd[0][2]
lengths_pcd = [length_A_x, length_A_y, length_A_z]

# Get the axis-aligned bounding box (AABB) for mesh B
aabb_original_mesh = original_mesh.bounds
length_B_x = aabb_original_mesh[1][0] - aabb_original_mesh[0][0]
length_B_y = aabb_original_mesh[1][1] - aabb_original_mesh[0][1]
length_B_z = aabb_original_mesh[1][2] - aabb_original_mesh[0][2]
lengths_original = [length_B_x, length_B_y, length_B_z]


scale_to = 0 #x: 0 y: 1 z: 2 all: 3

if scale_to == 0:
    scale = length_A_x / length_B_x
elif scale_to == 1:
    scale = length_A_y / length_B_y
elif scale_to == 2:
    scale = length_A_z / length_B_z
else:
    scale = np.divide(lengths_pcd, lengths_original)

#scale = length_A_x / length_B_x #x axis
#scale = length_A_y / length_B_y #y axis
#scale = length_A_z / length_B_z #z axis

print(lengths_pcd, lengths_original)
print(original_mesh.extents)
print(mesh.extents)
print(scale)

#show both meshes

original_mesh.apply_scale(scale)
original_mesh_scene = original_mesh.scene()
original_mesh_scene.add_geometry(trimesh.creation.axis(origin_size=.01))
original_mesh_scene.add_geometry(mesh)
original_mesh_scene.show()


# original_mesh.show()

In [ ]:
print(original_mesh.bounds[1] - original_mesh.bounds[0])
print(mesh.bounds[1] - mesh.bounds[0])

In [ ]:
scale_to_dict = {
    0: "x",
    1: "y",
    2: "z",
    3: "all"
}

scale_to_str = scale_to_dict[scale_to]

mesh_name = f"scaled_from_{depth_estimator}_erode_{erode_iter}_scale_{scale_to_str}"
#create new folder for the scaled mesh
os.makedirs(os.path.join(original_mesh_parent_dir, mesh_name), exist_ok=True)

save_loc = os.path.join(original_mesh_parent_dir, mesh_name, mesh_name + ".obj")
original_mesh.export(save_loc)
print("saved to", save_loc)


In [ ]:
#estimate distance of obj from camera
tgt_obj_depths = depth[masks[0] == tgt_obj]

#print(np.unique(tgt_obj_depths))

#get top 5 and bottom 5
top_5 = np.sort(tgt_obj_depths)[-5:]
bottom_5 = np.sort(tgt_obj_depths)[:5]
print(f"Top 5 depths: {top_5}")
print(f"Bottom 5 depths: {bottom_5}")
print(f"Mean depth {np.mean(tgt_obj_depths)}")
print(f"Median depth {np.median(tgt_obj_depths)}")

# display a histogram of depths, removing outliers
tgt_obj_depths = tgt_obj_depths[tgt_obj_depths < 10]
plt.hist(tgt_obj_depths, bins=20)
pass

In [ ]:
obj_name = original_mesh_loc.split("/")[-1].split(".")[0]
start_dist = .9
end_dist = 1.2

tracking_command = [
    "conda activate foundationpose &&",
    "CUDA_VISIBLE_DEVICES=5",
    "python third_party/FoundationPose_austin/run_demo.py",
    f"--mesh_file {save_loc}",
    f"--test_scene_dir {annotation_dir}/FoundationPose",
    "--has_depth 0",
    f"--dist_to_cam '[{start_dist}, {end_dist}]'",
    "--track_refine_iter 100",
    "--debug 3",
    f"--experiment_name {obj_name}_InstantMesh_metric3d_scaling_{scale_to_str}_100",
    "--use_mmde True",
    "--bs_scale .25",
    f"--mask_index {tgt_obj}",
]

est_command = [
    "conda activate foundationpose &&",
    "CUDA_VISIBLE_DEVICES=7",
    "python third_party/FoundationPose_austin/run_demo.py",
    f"--mesh_file {save_loc}",
    f"--test_scene_dir {annotation_dir}/FoundationPose",
    "--has_depth 0",
    f"--dist_to_cam '[{start_dist}, {end_dist}]'",
    "--est_refine_iter 10",
    "--track_refine_iter 100",
    "--debug 3",
    f"--experiment_name {obj_name}_InstantMesh_metric3d_scaling_{scale_to_str}_est_only",
    "--estimate_only True",
    "--frame_sample_rate 5",
    "--use_mmde True",
    "--bs_scale .25",
    f"--mask_index {tgt_obj}",
]

print("Tracking command: ")
print(" ".join(tracking_command), "\n\n\n")
print("Estimation command: ")
print(" ".join(est_command))
